<!-- launch-badges -->
<a href="https://colab.research.google.com/github/laban254/ml-for-infrastructure/blob/main/01_foundations/distributed_data/pyspark_log_processing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
&nbsp;
<a href="https://mybinder.org/v2/gh/laban254/ml-for-infrastructure/main?urlpath=lab/tree/01_foundations/distributed_data/pyspark_log_processing.ipynb" target="_blank"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"/></a>

> ▶️ **Run this notebook live** — no install needed. Click a badge above to open it in a free cloud runtime.
>
> 🖥️ **Running locally instead?** Select a Jupyter kernel backed by an environment that has `pyspark` installed (e.g. this repo's `.venv`). Picking the wrong interpreter is the most common cause of `ModuleNotFoundError: No module named 'pyspark'`.

# Distributed Data Processing with PySpark

## Context
Production log and metrics pipelines routinely generate volumes that no single machine can hold in memory or process quickly enough — think years of ELK stack exports or metrics scraped every few seconds across thousands of hosts. PySpark lets you write familiar DataFrame-style code that Spark transparently splits into parallel tasks across cores or an entire cluster, so the same query that groups a few thousand rows in Pandas can scale to billions. Note that this notebook is a simplified local demo: for teaching purposes we first build the synthetic dataset (500k rows by default — see the quick/full mode toggle in Section 2) as a Pandas DataFrame before converting it to Spark, whereas a real pipeline would generate or read the data directly as a Spark DataFrame from distributed storage (e.g. S3, HDFS) rather than materializing it in a single process's memory first.

## Objectives
- Bridge the gap between local tools (Pandas) and infrastructure-level big data processing (`PySpark`).
- Understand the map-reduce paradigm for processing logs or metrics that won't fit into your local machine's RAM.
- Set up a local Spark Session to aggregate and query dataset sizes that typically require a cluster.

## Expected Outcome
- A functional local PySpark pipeline capable of grouping and extracting metrics from millions of log rows.

## Challenge
- Rewrite a standard Pandas DataFrame `groupby()` utilizing PySpark RDDs or Spark DataFrames.

In [1]:
# !pip install pyspark

### 1. Initializing Spark
Unlike Pandas, Spark requires an active "Session" or "Context" that connects your Python code to the JVM (Java Virtual Machine) backend that does the heavy lifting.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, count, avg

# Set up a local spark session using all available CPU cores (*)
spark = SparkSession.builder \
    .appName("InfraLogAnalysis") \
    .master("local[*]") \
    .getOrCreate()

# Suppress verbose WARN/progress-bar noise so aggregation output below stays readable
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/16 13:01:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 2. Simulating a Gigantic Log File
We'll generate a dummy dataset of server events that resembles a real-world ELK stack export.

In [3]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

np.random.seed(42)  # reproducible synthetic logs

# notebook_toggle.py lives at the repo root; walk up from this notebook's own
# directory to find it, since Jupyter sets the kernel's cwd to the notebook's
# folder, not wherever `jupyter lab` was launched from.
try:
    _repo_root = next(p for p in Path.cwd().resolve().parents if (p / "notebook_toggle.py").exists())
    sys.path.insert(0, str(_repo_root))
    from notebook_toggle import get_mode
    # notebook_toggle.py defaults to "quick" when NOTEBOOK_MODE is unset, but
    # this notebook exists to demonstrate distributed-scale processing, so it
    # only goes quick when explicitly asked — full is the sensible default.
    quick_mode = "NOTEBOOK_MODE" in os.environ and get_mode() == "quick"
except (ImportError, StopIteration):
    # notebook_toggle.py isn't reachable (e.g. a Colab session that only
    # fetched this single file) — fall back to the full-scale run.
    quick_mode = False

num_records = 50_000 if quick_mode else 500_000
print(f"Mode: {'quick' if quick_mode else 'full'}  ->  generating {num_records:,} synthetic log records")
sample_services = ['auth-service', 'billing-api', 'frontend-ui', 'database-pg']
sample_levels = ['INFO', 'WARN', 'ERROR', 'DEBUG']

data = {
    "timestamp": pd.date_range(start="2025-01-01", periods=num_records, freq="15s"),
    "service": np.random.choice(sample_services, num_records, p=[0.4, 0.2, 0.3, 0.1]),
    "log_level": np.random.choice(sample_levels, num_records, p=[0.7, 0.1, 0.05, 0.15]),
    "response_time_ms": np.random.gamma(shape=2.0, scale=50.0, size=num_records)
}

# Convert Pandas DataFrame to PySpark DataFrame
pdf = pd.DataFrame(data)
df = spark.createDataFrame(pdf)

print("Spark DataFrame Schema:")
df.printSchema()

Mode: full  ->  generating 500,000 synthetic log records


Spark DataFrame Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- service: string (nullable = true)
 |-- log_level: string (nullable = true)
 |-- response_time_ms: double (nullable = true)



### 3. Distributed Queries (Lazy Evaluation)
In Spark, defining a query doesn't execute it immediately. Execution happens only when an action (like `.show()` or `.collect()`) is called.

In [4]:
# Aggregating average response times and error counts by service
# If this was a 100GB dataset, this query would farm out to your cluster implicitly
service_metrics = df.filter(col("log_level").isin(["ERROR", "WARN"])) \
    .groupBy("service", "log_level") \
    .agg(
        count("*").alias("total_occurrences"),
        avg("response_time_ms").alias("avg_latency_ms")
    ) \
    .orderBy("total_occurrences", ascending=False)

print("Top Services with Warnings and Errors:")
service_metrics.show()

Top Services with Warnings and Errors:


+------------+---------+-----------------+------------------+
|     service|log_level|total_occurrences|    avg_latency_ms|
+------------+---------+-----------------+------------------+
|auth-service|     WARN|            20103| 99.95982366497164|
| frontend-ui|     WARN|            14969| 99.23865509749942|
|auth-service|    ERROR|            10066| 99.83245553213543|
| billing-api|     WARN|             9979|100.53756956876506|
| frontend-ui|    ERROR|             7528| 99.87938033889765|
| billing-api|    ERROR|             5101| 98.68337668568532|
| database-pg|     WARN|             4957|100.33525840275226|
| database-pg|    ERROR|             2479|  97.8750422568951|
+------------+---------+-----------------+------------------+



### 4. The Challenge: a Pandas `groupby()` rewritten in Spark

The intro set this as the challenge, so here it is done explicitly — **the same aggregation written twice**, once eagerly in Pandas and once lazily in Spark.

The point isn't that Spark is faster at this size (it isn't — a dataset this size fits comfortably in RAM, and Spark's coordination overhead dominates). The point is *where the time goes*: building a Spark query costs nothing, because nothing runs until an **action** like `.collect()` or `.show()` forces it.

In [5]:
import time

# --- Pandas: eager. The whole frame must fit in this process's RAM. ---
t0 = time.time()
pandas_result = (
    pdf[pdf["log_level"].isin(["ERROR", "WARN"])]
    .groupby(["service", "log_level"])
    .agg(
        total_occurrences=("service", "count"),
        avg_latency_ms=("response_time_ms", "mean"),
    )
    .sort_values("total_occurrences", ascending=False)
)
pandas_secs = time.time() - t0

# --- Spark: lazy. This only builds a query plan. ---
t0 = time.time()
spark_query = (
    df.filter(col("log_level").isin(["ERROR", "WARN"]))
    .groupBy("service", "log_level")
    .agg(
        count("*").alias("total_occurrences"),
        avg("response_time_ms").alias("avg_latency_ms"),
    )
    .orderBy("total_occurrences", ascending=False)
)
build_secs = time.time() - t0

# --- The action. This is what actually triggers the distributed job. ---
t0 = time.time()
spark_result = spark_query.collect()
collect_secs = time.time() - t0

print(f"Pandas  - eager groupby   : {pandas_secs:.3f}s")
print(f"Spark   - build the plan  : {build_secs:.3f}s   <- no data touched yet")
print(f"Spark   - .collect()      : {collect_secs:.3f}s   <- the whole job runs here")

# Align both results on (service, log_level) and compare the actual numbers,
# not just how many groups each side produced.
spark_aligned = (
    pd.DataFrame(spark_result, columns=["service", "log_level", "total_occurrences", "avg_latency_ms"])
    .set_index(["service", "log_level"])
    .sort_index()
)
pandas_aligned = pandas_result.sort_index()
counts_match = (spark_aligned["total_occurrences"] == pandas_aligned["total_occurrences"]).all()
latencies_match = np.allclose(spark_aligned["avg_latency_ms"], pandas_aligned["avg_latency_ms"])

print(f"\nSame result? counts match={counts_match}, avg latencies match={latencies_match} "
      f"({len(spark_aligned)} service/level groups both ways)")


[Stage 3:======================>                                    (3 + 5) / 8]



Pandas  - eager groupby   : 0.133s
Spark   - build the plan  : 0.077s   <- no data touched yet
Spark   - .collect()      : 3.040s   <- the whole job runs here

Same result? counts match=True, avg latencies match=True (8 service/level groups both ways)


Because the plan is built before anything executes, Spark gets to *optimise the whole query at once* — pushing the filter down to the scan so excluded rows are never materialised. That is exactly the optimisation you cannot express in eager Pandas. Here is the plan Spark actually chose:

In [7]:
# The physical plan Spark will execute - read it bottom-up.
# Note the filter appearing next to the scan, not after it.
spark_query.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(3) Sort [total_occurrences#50L DESC NULLS LAST], true, 0
   +- AQEShuffleRead coalesced
      +- ShuffleQueryStage 1
         +- Exchange rangepartitioning(total_occurrences#50L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=118]
            +- *(2) HashAggregate(keys=[service#1, log_level#2], functions=[count(1), avg(response_time_ms#3)])
               +- AQEShuffleRead coalesced
                  +- ShuffleQueryStage 0
                     +- Exchange hashpartitioning(service#1, log_level#2, 200), ENSURE_REQUIREMENTS, [plan_id=94]
                        +- *(1) HashAggregate(keys=[service#1, log_level#2], functions=[partial_count(1), partial_avg(response_time_ms#3)])
                           +- *(1) Project [service#1, log_level#2, response_time_ms#3]
                              +- *(1) Filter log_level#2 IN (ERROR,WARN)
                                 +- *(1) Scan ExistingRDD[timestamp#0,ser

## 📝 Exercise: p95 latency per service

Averages hide the tail, and the tail is what pages you. **Compute the 95th-percentile `response_time_ms` for each service** and print them worst-first.

Exact percentiles are expensive across a cluster (they need a global sort), so Spark ships an approximate version built for scale.

```python
# your code here
# hint: from pyspark.sql.functions import percentile_approx
```

> Run this **before** the `spark.stop()` cell below — once the session is stopped, `df` is no longer queryable.

<details><summary>💡 Reveal solution</summary>

```python
from pyspark.sql.functions import percentile_approx

(
    df.groupBy("service")
    .agg(
        percentile_approx("response_time_ms", 0.95).alias("p95_latency_ms"),
        avg("response_time_ms").alias("avg_latency_ms"),
        count("*").alias("events"),
    )
    .orderBy("p95_latency_ms", ascending=False)
    .show()
)
```

Compare the p95 and average columns: the gap between them is the part of your latency distribution that SLO dashboards built on averages will never show you.
</details>

## Key takeaways

| Concept | What you saw |
| --- | --- |
| **Lazy evaluation** | Building `spark_query` was ~instant; `.collect()` did all the work |
| **Actions vs transformations** | `filter`/`groupBy`/`agg` are transformations; `.collect()`/`.show()` are actions |
| **Query planning** | `.explain()` shows Spark pushing the filter down to the scan |
| **When Spark wins** | Not at this demo's scale — at volumes that exceed one machine's RAM |
| **Same mental model** | DataFrame code you already know from Pandas, executed in parallel |

Spark earns its overhead when the data no longer fits on one box. Below that line, Pandas is the right tool — knowing *where* that line sits is the actual skill.

In [7]:
# Stop the Spark context to free up memory
spark.stop()